# Seminar 2. Custom PyTorch Operators

# Building Models in PyTorch Through Composition

PyTorch models are built using **composition**.  
Instead of defining one large monolithic network, we construct models by combining smaller, reusable modules.

Each module can contain other modules, which allows us to build hierarchical and well-structured architectures.

---

## Composition

Composition means:

- A model is built from smaller blocks.
- Each block can contain multiple layers.
- Blocks can be reused in larger architectures.
- Complex models are created by stacking simpler components.

This keeps code:

- Modular  
- Reusable  
- Readable  
- Easy to extend  


## Key Ideas

- Inherit from `nn.Module`
- Define layers inside `__init__`
- Define computation in `forward()`
- Create reusable blocks
- Build larger models by combining blocks

---

## Example: Model Built from Two Blocks

Below is a simple example where:

- We define a reusable blocks: `LinearReLUBlock` and `LinearTanhBlock`
- The final model is composed of two such blocks


In [ ]:
import torch
import torch.nn as nn


class LinearReLUBlock(nn.Module):
    def __init__(self, in_features: int, out_features: int):
        super().__init__()

        self.linear = nn.Linear(in_features, out_features)
        self.activation = nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.linear(x)
        x = self.activation(x)
        return x


class LinearTanhBlock(nn.Module):
    def __init__(self, in_features: int, out_features: int):
        super().__init__()

        self.linear = nn.Linear(in_features, out_features)
        self.activation = nn.Tanh()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.linear(x)
        x = self.activation(x)
        return x


class CombinedModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.block1 = LinearReLUBlock(4, 8)
        self.block2 = LinearTanhBlock(8, 8)
        self.output = nn.Linear(8, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.block1(x)
        x = self.block2(x)
        x = self.output(x)
        return x


model = CombinedModel()
print(model)

# What `nn.Module` Enables

When we inherit from `nn.Module`, we automatically gain powerful functionality that works **recursively across all submodules**.

## What `nn.Module` Gives Us



### Parameter Registration

All layers assigned as attributes (e.g. `self.linear = nn.Linear(...)`) are:

- Automatically registered
- Collected in `model.parameters()`
- Included in `model.state_dict()`

This works **recursively** for all sub-blocks.

In [ ]:
print("Registered parameters:")
for name, param in model.named_parameters():
    print(name, param.shape)


### Automatic Gradient Tracking

During the forward pass:

- PyTorch dynamically builds a computation graph
- Calling `loss.backward()` computes gradients
- Gradients are stored in each parameter’s `.grad`

No manual graph management is required.

In [ ]:
x = torch.randn(5, 4)
target = torch.randn(5, 2)

criterion = nn.MSELoss()
output = model(x)
loss = criterion(output, target)

loss.backward()

print("\nGradient computed for output layer:",
      model.output.weight.grad is not None)
model.output.weight.grad

NameError: name 'torch' is not defined

### Device and Type Transfer (`.to()`)

Calling:

    model.to(device)

or

    model.to(dtype)

moves all:

- Parameters
- Buffers
- Submodules

to CPU/GPU/dtype automatically.

In [ ]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

dtype = torch.float32

model = model.to(device=device, dtype=dtype)

x: torch.Tensor = torch.randn(5, 4, device=device, dtype=dtype)

print("Model device:", next(model.parameters()).device)
print("Model dtype:", next(model.parameters()).dtype)

### Saving & Loading (`state_dict()`)

- `model.state_dict()` returns all parameters recursively
- `model.load_state_dict(...)` restores them

This works across the full module tree.

In [ ]:
state_dict = model.state_dict()
torch.save(state_dict, "combined_model.pt")

# `train()` vs `eval()` Mode in PyTorch

PyTorch modules have two main modes: **training mode** and **evaluation mode**.  
Switching between them affects layers that behave differently during training and inference.

---

## `model.train()`

- Sets the model to **training mode**.
- Used when training the model with gradient updates.
- Affects certain layers, such as:

| Layer Type        | Behavior in `train()` Mode                  |
|------------------|--------------------------------------------|
| `Dropout`         | Randomly zeroes some activations           |
| `BatchNorm`       | Updates running statistics (mean/variance) |

- Gradients are computed as usual.

---

## `model.eval()`

- Sets the model to **evaluation (inference) mode**.
- Used when evaluating or deploying the model.
- Affects certain layers:

| Layer Type        | Behavior in `eval()` Mode                   |
|------------------|--------------------------------------------|
| `Dropout`         | Passes all activations through unchanged  |
| `BatchNorm`       | Uses stored running mean/variance         |

- No layers update internal statistics.
- Gradients are usually not required (often used with `torch.no_grad()`).

---

## Key Points

- Always use `model.train()` during training.
- Always use `model.eval()` during evaluation or testing.
- Forgetting to switch can lead to inconsistent results, especially with `Dropout` or `BatchNorm`.



In [ ]:
import torch
import torch.nn as nn

# Simple model with Dropout and BatchNorm
class SimpleModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.fc1 = nn.Linear(4, 8)
        self.bn = nn.BatchNorm1d(8)
        self.dropout = nn.Dropout(p=0.5)
        self.fc2 = nn.Linear(8, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.fc1(x)
        x = self.bn(x)
        x = torch.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x


model = SimpleModel()
x = torch.randn(5, 4)

# Training mode
model.train()
output_train = model(x)
print("Training mode output:\n", output_train)

# Evaluation mode
model.eval()
with torch.no_grad():
    output_eval = model(x)
print("Evaluation mode output:\n", output_eval)


# `torch.no_grad()` and `torch.inference_mode()` in PyTorch

When performing inference (evaluating a model without updating parameters), PyTorch provides context managers to **disable gradient tracking**. This saves memory and speeds up computation.

---

## `torch.no_grad()`

- Disables gradient tracking.
- Useful during evaluation or inference.
- Gradients are **not computed**, but autograd still tracks operations for some internal purposes.
- Can be used as a **context manager** or a **function decorator**.

---

## `torch.inference_mode()`

- Introduced in PyTorch 1.9.
- Similar to `no_grad()`, but **more efficient**.
- Completely disables autograd and reduces memory usage.
- Recommended for pure inference pipelines.



In [ ]:
import torch
import torch.nn as nn

class SimpleModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.fc = nn.Linear(4, 2)

    # -----------------------------
    # Using torch.no_grad() as method decorator
    # -----------------------------
    @torch.no_grad()
    def forward_no_grad(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)

    # -----------------------------
    # Using torch.inference_mode() as method decorator
    # -----------------------------
    @torch.inference_mode()
    def forward_inference(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)


model = SimpleModel()
x = torch.randn(5, 4)

# -----------------------------
# Call decorated methods
# -----------------------------
output_no_grad_method = model.forward_no_grad(x)
output_infer_method = model.forward_inference(x)

print("Output no_grad method:\n", output_no_grad_method)
print("Output inference_mode method:\n", output_infer_method)

# -----------------------------
# Using context managers
# -----------------------------

with torch.no_grad():
    output_no_grad_cm = model(x)

with torch.inference_mode():
    output_infer_cm = model(x)

print("Output no_grad context manager:\n", output_no_grad_cm)
print("Output inference_mode context manager:\n", output_infer_cm)


# Disabling Gradients with `requires_grad_(False)`

PyTorch provides a convenient method `requires_grad_()` that can **enable or disable gradients in-place** for all parameters of a model or a tensor.

Using:

```python
param.requires_grad_(False)
```

- Sets `requires_grad=False` **in-place** for that parameter.
- This is useful for freezing models during inference or transfer learning.
- Can be applied to an entire model recursively by iterating over its parameters.


In [ ]:
model = SimpleModel()

# Disable gradient computation for all parameters using requires_grad_()
for param in model.parameters():
    param.requires_grad_(False)

# Verify
for name, param in model.named_parameters():
    print(f"{name}: requires_grad={param.requires_grad}")

# Forward pass still works
x = torch.randn(5, 4)
output = model(x)
print("Output shape:", output.shape)

# Redefining `train()` and `eval()` in `nn.Module`

PyTorch’s `nn.Module` provides built-in `train(mode: bool = True)` and `eval()` methods to switch between **training** and **evaluation** modes.  

Sometimes, when creating **custom modules or blocks**, you might want to **override these methods** to perform extra actions whenever the mode changes.

---

## Why Override?

- Apply mode-specific logic to sub-blocks or attributes that are not standard layers
- Log or track mode switches
- Automatically modify internal flags or buffers along with training/eval mode

---

## How It Works

- `train(mode: bool = True)` sets `self.training = mode` for the module
- `eval()` is equivalent to `train(False)`
- Default implementation recursively calls `train(mode)` on all submodules
- Overriding allows custom behavior while keeping recursive updates intact


In [ ]:
import torch
import torch.nn as nn
from torch import Tensor
from typing import Self

class CustomBlock(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.linear = nn.Linear(4, 4)
        self.dropout = nn.Dropout(p=0.5)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.linear(x)
        x = torch.relu(x)
        x = self.dropout(x)
        return x

    # -----------------------------
    # Override train() method
    # -----------------------------
    def train(self, mode: bool = True) -> Self:
        print(f"CustomBlock set to {'train' if mode else 'eval'} mode")
        super().train(mode)  # Call original method to update submodules
        # Add any custom logic here
        return self

    # -----------------------------
    # Override eval() method
    # -----------------------------
    def eval(self) -> Self:
        print("CustomBlock set to eval mode")
        return super().eval()


# Example usage
model = CustomBlock()
x = torch.randn(2, 4)

# Switch to training mode
model.train()
output_train = model(x)

# Switch to evaluation mode
model.eval()
with torch.no_grad():
    output_eval = model(x)

print("Output training mode:", output_train)
print("Output eval mode:", output_eval)


# Common Module Aggregators in PyTorch

When building neural networks, it is often useful to group multiple layers or submodules together.  
PyTorch provides several **module aggregators** that help organize layers and blocks. The most common ones are:



## `nn.Sequential`

- Holds modules in a sequential order.
- Executes them **in the order they are added** during the forward pass.
- Ideal for simple **stacked layers** with a single input and output.

**Key points:**

- Forward pass is automatically defined.
- Cannot handle multiple inputs or branching.

In [ ]:
seq_model = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    nn.Linear(8, 2)
)

x = torch.randn(5, 4)
output_seq = seq_model(x)
print("nn.Sequential output shape:", output_seq.shape)


## `nn.ModuleList`

- Holds a **list of modules**.
- Does **not define a forward pass automatically**.
- Useful when you need to **loop over modules**, or have conditional computation.

**Key points:**

- Modules are registered properly, so parameters are tracked.
- You must define your own `forward()`.

In [ ]:
class ModuleListModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.layers = nn.ModuleList([
            nn.Linear(4, 8),
            nn.ReLU(),
            nn.Linear(8, 2)
        ])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x)
        return x

ml_model = ModuleListModel()
output_ml = ml_model(x)
print("nn.ModuleList output shape:", output_ml.shape)

## `nn.ModuleDict`

- Holds modules in a **dictionary** with string keys.
- Useful for architectures with **named branches**, **dynamic selection**, or **multi-head outputs**.
- Like `ModuleList`, it does **not define a forward pass**.

In [ ]:
class ModuleDictModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.branches = nn.ModuleDict({
            "branch1": nn.Linear(4, 8),
            "branch2": nn.Linear(4, 8)
        })
        self.output: nn.Linear = nn.Linear(8, 2)

    def forward(self, x: torch.Tensor, branch_name: str = "branch1") -> torch.Tensor:
        x = self.branches[branch_name](x)
        return self.output(x)

md_model = ModuleDictModel()
output_md = md_model(x, branch_name="branch2")
print("nn.ModuleDict output shape:", output_md.shape)


## Работа на семинаре

### LSTM

![lstm](assets/LSTM.png)

In [ ]:
import torch
from typing import Tuple
import torch.nn.functional as F

class LSTMCell(nn.Module):
  def __init__(self, input_size: int, hidden_size: int, bias: bool = False):
    super().__init__()
    self.input_size = input_size
    self.hidden_size = hidden_size

    self.x_gates = nn.Linear(input_size, hidden_size*4, bias=bias)
    self.h_gates = nn.Linear(hidden_size, hidden_size*4, bias=bias)

  def forward(self, x: torch.Tensor, h_prev: torch.Tensor, c_prev: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    '''
    Input:
      x: [batch_size, input_size]
      h_t-1: [batch_size, hidden_size]
      c_t-1: [batch_size, hidden_size]

    Output:
      h_t: [batch_size, hidden_size]
      c_t: [batch_size, hidden_size]
    '''
    gates_out = self.x_gates(x) + self.h_gates(h_prev)
    f_gate, c_gate, i_gate, o_gate = torch.chunk(gates_out, 4, -1)
    f_gate = F.sigmoid(f_gate)
    c_gate = F.tanh(c_gate)
    i_gate = F.sigmoid(i_gate)
    o_gate = F.sigmoid(o_gate)
    ci_gate = c_gate*i_gate
    c_t = f_gate*c_prev + ci_gate
    h_t = F.tanh(c_t)*o_gate

    return c_t, h_t


h = torch.rand(2,4)
c = torch.rand(2,4)
x = torch.rand(2,2)

lstm = LSTMCell(2,4)
h = torch.zeros(2,4)
c = torch.zeros(2,4)
x_lst = [torch.rand(2,4) for _ in range(5)]

for x in x_lst:
  h, c = lstm(x,h,c)

### Inception

![inception](assets/inception.png)

### SE

![se](assets/SqueezeAndExcite.png)

### Selective Kernel

![selective](assets/SelectiveKernel.png)


### PatchMerger

![patchmerger](assets/PatchMerger.png)


## Homework

2 задания:
1. Реализуйте требуемый в заголовке блок (максмсум 0.8 балов).

## ResNet Block (0.1 балл)

![Resnet](assets/ResBlock.png)

https://arxiv.org/pdf/1512.03385

In [5]:
import torch
import torch.nn as nn

class ResidualBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, stride: int = 1):
        """        
        Args:
            in_channels (int): Number of input channels
            out_channels (int): Number of output channels  
            stride (int): Stride for the first convolution
        """
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 
                               kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        
        self.conv2 = nn.Conv2d(out_channels, out_channels, 
                               kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 
                         kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )


    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = x
        
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        
        out = self.conv2(out)
        out = self.bn2(out)
        
        identity = self.shortcut(identity)
        
        out += identity
        out = self.relu(out)
        return out
    
cnn_block = ResidualBlock(64, 32)
x = torch.randn(1, 64, 32, 32)
y = cnn_block(x)
print(f"Input shape: {x.shape}")
print(f"Output shape: {y.shape}")

Input shape: torch.Size([1, 64, 32, 32])
Output shape: torch.Size([1, 32, 32, 32])


## Depthwise Separable Convolution (0.1 балл)
![DepthWiseConv](assets/DepthWiseConv.png)

https://arxiv.org/pdf/1610.02357

In [6]:
class SeparableConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False):
        """        
        Args:
            in_channels (int): M - number of input channels
            out_channels (int): N - number of output channels
            kernel_size (int): k - spatial size of depthwise convolution kernel
            stride (int): stride for depthwise convolution
            padding (int): padding for depthwise convolution
            bias (bool): whether to use bias
        """
        super().__init__()
        self.depthwise = nn.Conv2d(
            in_channels, in_channels, kernel_size=kernel_size,
            stride=stride, padding=padding, groups=in_channels, bias=bias
        )
        self.pointwise = nn.Conv2d(
            in_channels, out_channels, kernel_size=1, stride=1, padding=0, bias=bias
        )


    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.depthwise(x)
        x = self.pointwise(x)
        return x

sep_conv2v = SeparableConv2d(64, 32)
x = torch.randn(1, 64, 32, 32)
y = sep_conv2v(x)
print(f"Input shape: {x.shape}")
print(f"Output shape: {y.shape}")

Input shape: torch.Size([1, 64, 32, 32])
Output shape: torch.Size([1, 32, 32, 32])


## Vanilla Attention (0.1 балл)

Let:

$$
\text{query} \in \mathbb{R}^{B \times d} \\
\text{key} \in \mathbb{R}^{B \times L \times d}
$$

---

### Alignment Scores

$$
\text{score} = \text{key} \cdot (W_\text{align} \, \text{query})^T \\
\text{score} \in \mathbb{R}^{B \times L}
$$

---

### Attention Weights

$$
\text{att} = \text{softmax}(\text{score}, \text{dim}=1) \\
\text{att} \in \mathbb{R}^{B \times L}
$$

---

### Context Vector

$$
\text{context} = \sum_{i=1}^{L} \text{att}_i \cdot \text{key}_i \\
\text{context} \in \mathbb{R}^{B \times d}
$$

---

### Output

$$
\text{out} = \tanh(W_\text{value} \, \text{context} + W_\text{query} \, \text{query}) \\
\text{out} \in \mathbb{R}^{B \times d}
$$



https://arxiv.org/abs/1409.0473


https://arxiv.org/abs/1508.04025

In [7]:
from typing import Optional
import torch
from torch import nn
import numpy as np

class VanillaAttention(nn.Module):
    def __init__(self, d_model: int, d_k: Optional[int] = None):
        """
        Args:
            d_model (int): Dimension of the model (d in the equations)
            d_k (int, optional): Dimension for alignment projection. If None, uses d_model
        """
        super().__init__()
        self.d_model = d_model
        self.d_k = d_k if d_k is not None else d_model
        self.W_align = nn.Linear(self.d_model, self.d_k, bias=False)
        self.W_value = nn.Linear(self.d_model, self.d_model, bias=False)
        self.W_query = nn.Linear(self.d_model, self.d_model, bias=False)


    def forward(self, query: torch.Tensor, key: torch.Tensor) -> torch.Tensor:
        projected_query = self.W_align(query)
        score = torch.bmm(key, projected_query.unsqueeze(-1)).squeeze(-1)
        att = torch.softmax(score, dim=1)
        context = torch.bmm(att.unsqueeze(1), key).squeeze(1)
        value_proj = self.W_value(context)
        query_proj = self.W_query(query)
        out = torch.tanh(value_proj + query_proj)
        return out
    
B = 4
L = 10
d = 64  
query = torch.randn(B, d)
key = torch.randn(B, L, d)

print(f"Query shape: {query.shape} (B={B}, d={d})")
print(f"Key shape: {key.shape} (B={B}, L={L}, d={d})")

attention = VanillaAttention(d_model=d)
output = attention(query, key)
print(f"Output shape: {output.shape} (B={B}, d={d})")

Query shape: torch.Size([4, 64]) (B=4, d=64)
Key shape: torch.Size([4, 10, 64]) (B=4, L=10, d=64)
Output shape: torch.Size([4, 64]) (B=4, d=64)


## Dot Product Attention (0.1 балл)

$$
Q \in \mathbb{R}^{B \times L_q \times d_k} \\
K \in \mathbb{R}^{B \times L_k \times d_k} \\
V \in \mathbb{R}^{B \times L_k \times d_k}
$$

$$
S = \frac{Q K^T}{\sqrt{d_k}}
$$

$$
\text{Attention}(Q, K, V) = \text{softmax}(S, \text{dim}=-1) \, V
$$



https://arxiv.org/abs/1706.03762


In [8]:
import torch
from torch import nn
import torch.nn.functional as F

class ScaledDotProductAttention(nn.Module):
    def __init__(self, dropout: float = 0.0):
        """        
        Args:
            dropout (float): Dropout probability for attention weights
        """
        super().__init__()
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()


    def forward(self, query: torch.Tensor, 
                key: torch.Tensor, 
                value: torch.Tensor,
                mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        """
        Args:
            query: Tensor of shape (B, L_q, d_k) - queries
            key: Tensor of shape (B, L_k, d_k) - keys
            value: Tensor of shape (B, L_k, d_v) - values
            mask: Optional mask tensor of shape (B, L_q, L_k) or broadcastable shape
            
        Returns:
            output: Tensor of shape (B, L_q, d_v) - attention output
        """
        d_k = query.shape[2]
        scores = torch.matmul(query, key.transpose(-2, -1)) / torch.sqrt(torch.tensor(d_k, dtype=torch.float32))
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        att_weights = F.softmax(scores, dim=-1)
        att_weights = self.dropout(att_weights)
        output = torch.matmul(att_weights, value)        
        return output


B = 4
L_q = 8
L_k = 10
d_k = 64 
d_v = 64 
query = torch.randn(B, L_q, d_k)
key = torch.randn(B, L_k, d_k)
value = torch.randn(B, L_k, d_v)

print(f"Query shape: {query.shape} (B={B}, L_q={L_q}, d_k={d_k})")
print(f"Key shape: {key.shape} (B={B}, L_k={L_k}, d_k={d_k})")
print(f"Value shape: {value.shape} (B={B}, L_k={L_k}, d_v={d_v})")

attention = ScaledDotProductAttention(dropout=0.1)
output = attention(query, key, value)
print(f"Output shape: {output.shape} (B={B}, L_q={L_q}, d_v={d_v})")

Query shape: torch.Size([4, 8, 64]) (B=4, L_q=8, d_k=64)
Key shape: torch.Size([4, 10, 64]) (B=4, L_k=10, d_k=64)
Value shape: torch.Size([4, 10, 64]) (B=4, L_k=10, d_v=64)
Output shape: torch.Size([4, 8, 64]) (B=4, L_q=8, d_v=64)


## Multihead Attention (0.1 балл)

![MultiheadAttention](assets/MultiheadAttention.webp)

https://arxiv.org/abs/1706.03762


In [11]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.0):
        """
        Args:
            d_model: Model dimension
            num_heads: Number of attention heads
            dropout: Dropout probability
        """
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)

        self.attention = ScaledDotProductAttention(dropout=dropout)
        self.W_o = nn.Linear(d_model, d_model, bias=False)


    def forward(self, query: torch.Tensor, 
                key: torch.Tensor, 
                value: torch.Tensor,
                mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        """
        Args:
            query: (B, L_q, d_model)
            key: (B, L_k, d_model)
            value: (B, L_k, d_model)
            mask: Optional mask
            
        Returns:
            output: (B, L_q, d_model)
        """
        B, L_q, _ = query.shape
        B, L_k, _ = key.shape

        Q = self.W_q(query)
        K = self.W_k(key)
        V = self.W_v(value)  

        Q = Q.view(B, L_q, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(B, L_k, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(B, L_k, self.num_heads, self.d_k).transpose(1, 2) 

        if mask is not None:
            mask = mask.unsqueeze(1)
        
        attn_output = self.attention(Q, K, V, mask)
        attn_output = attn_output.transpose(1, 2).contiguous().view(B, L_q, self.d_model)
        output = self.W_o(attn_output)
        return output
    
B = 4
L_q = 8
L_k = 10
d_k = 64 
d_v = 64 
query = torch.randn(B, L_q, d_k)
key = torch.randn(B, L_k, d_k)
value = torch.randn(B, L_k, d_v)

print(f"Query shape: {query.shape} (B={B}, L_q={L_q}, d_k={d_k})")
print(f"Key shape: {key.shape} (B={B}, L_k={L_k}, d_k={d_k})")
print(f"Value shape: {value.shape} (B={B}, L_k={L_k}, d_v={d_v})")

attention = MultiHeadAttention(d_model=d_k, num_heads=4)
output = attention(query, key, value)
print(f"Output shape: {output.shape} (B={B}, L_q={L_q}, d_v={d_v})")

Query shape: torch.Size([4, 8, 64]) (B=4, L_q=8, d_k=64)
Key shape: torch.Size([4, 10, 64]) (B=4, L_k=10, d_k=64)
Value shape: torch.Size([4, 10, 64]) (B=4, L_k=10, d_v=64)
Output shape: torch.Size([4, 8, 64]) (B=4, L_q=8, d_v=64)


## Transformer Encoder Layer (0.1 балл)


![Transformer Encoder Layer](assets/TransformerEncoder.png)


https://arxiv.org/abs/1706.03762

In [ ]:
from typing import Optional
import torch
import torch.nn as nn
import torch.nn.functional as F

class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model: int = 512, 
                 nhead: int = 8, 
                 dim_feedforward: int = 2048, 
                 dropout: float = 0.1,
                 layer_norm_eps: float = 1e-5,
                 bias: bool = True):
        """        
        Args:
            d_model: Input dimension (T_N)
            nhead: Number of attention heads
            dim_feedforward: Dimension of feedforward network (MLP hidden dimension)
            dropout: Dropout probability
            layer_norm_eps: Epsilon for layer normalization
            bias: Whether to use bias in linear layers
        """
        super().__init__()
        self.d_model = d_model
        self.nhead = nhead
        
        self.self_attn = MultiHeadAttention(
            d_model, nhead, dropout=dropout
        )

        self.mlp = nn.Sequential(
            nn.Linear(d_model, dim_feedforward, bias=bias),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim_feedforward, d_model, bias=bias),
            nn.Dropout(dropout)
        )

        self.norm1 = nn.LayerNorm(d_model, eps=layer_norm_eps)
        self.norm2 = nn.LayerNorm(d_model, eps=layer_norm_eps)


    def forward(self, 
                src: torch.Tensor, 
                src_mask: Optional[torch.Tensor] = None,) -> torch.Tensor:
        """        
        Args:
            src: Input tensor of shape (batch_size, seq_len, d_model) - T_N
            src_mask: Optional attention mask of shape (seq_len, seq_len) or (batch_size, seq_len, seq_len)
            
        Returns:
            Output tensor of shape (batch_size, seq_len, d_model) - T_{N+1}
        """
        src_norm = self.norm1(src)
        attn_out = self.self_attn(
            src_norm, src_norm, src_norm,
            mask=src_mask
        )
        src = src + attn_out

        src_norm = self.norm2(src)
        ff_out = self.mlp(src_norm)
        src = src + ff_out
        
        return src
    
batch_size = 4
seq_len = 10
d_model = 512
nhead = 8

x = torch.randn(batch_size, seq_len, d_model)
print(f"Input shape (T_N): {x.shape}")
print(f"  batch_size: {batch_size}, seq_len: {seq_len}, d_model: {d_model}")

encoder_layer = TransformerEncoderLayer(
        d_model=d_model,
        nhead=nhead,
        dim_feedforward=2048,
    )

output = encoder_layer(x)
print(f"Output shape (T_N+1): {output.shape}")

Input shape (T_N): torch.Size([4, 10, 512])
  batch_size: 4, seq_len: 10, d_model: 512
Output shape (T_N+1): torch.Size([4, 10, 512])


## MLP Mixer (0.1 балл)


![MLPMixer](assets/MLPMixer.png)


https://arxiv.org/abs/2105.01601

In [21]:
import torch
import torch.nn as nn

class MLPMixerBlock(nn.Module):
    def __init__(self, 
                 num_patches: int,
                 hidden_dim: int,
                 tokens_mlp_dim: int,
                 channels_mlp_dim: int,
                 layer_norm_eps: float = 1e-5):
        """
        Args:
            num_patches (int): Number of patches (S in paper, T in diagram)
            hidden_dim (int): Hidden dimension per patch (C in diagram)
            tokens_mlp_dim (int): Token-mixing MLP hidden dimension (M in diagram)
            channels_mlp_dim (int): Channel-mixing MLP hidden dimension (P in diagram)
            layer_norm_eps (float): Epsilon for layer normalization
        """
        super().__init__()
        self.num_patches = num_patches
        self.hidden_dim = hidden_dim

        self.norm1 = nn.LayerNorm(hidden_dim, eps=layer_norm_eps)
        self.norm2 = nn.LayerNorm(hidden_dim, eps=layer_norm_eps)
        
        self.token_mixing = nn.Sequential(
            nn.Linear(num_patches, tokens_mlp_dim),
            nn.GELU(),
            nn.Linear(tokens_mlp_dim, num_patches),
        )

        self.channel_mixing = nn.Sequential(
            nn.Linear(hidden_dim, channels_mlp_dim),
            nn.GELU(),
            nn.Linear(channels_mlp_dim, hidden_dim),
        )


    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = x
        x = self.norm1(x)
        
        x = x.transpose(1, 2) 
        x = self.token_mixing(x)
        x = x.transpose(1, 2) 
        x = residual + x

        residual = x
        x = self.norm2(x)

        x = self.channel_mixing(x)
        x = residual + x
        
        return x

B = 4
T = 196
C = 512
M = 256
P = 2048 

x = torch.randn(B, T, C)
print(f"Input shape (B, T, C): {x.shape}")

mixer_block = MLPMixerBlock(
    num_patches=T,
    hidden_dim=C,
    tokens_mlp_dim=M,
    channels_mlp_dim=P
)

output = mixer_block(x)
print(f"Output shape: {output.shape}")

Input shape (B, T, C): torch.Size([4, 196, 512])
Output shape: torch.Size([4, 196, 512])


## ConvMixer (0.1 балл)

![ConvMixer](assets/ConvMixer.png)


https://arxiv.org/abs/2201.09792

In [22]:
class ConvMixer(nn.Module):

    def __init__(self, 
                 dim: int,
                 depthwise_kernel_size: int = 3):
        """
        Args:
            dim: Number of channels (dimension)
            depthwise_kernel_size: Kernel size for depthwise convolution
            dropout: Dropout rate
        """
        super().__init__()
        self.depthwise = nn.Sequential(
            nn.Conv2d(dim, dim, kernel_size=depthwise_kernel_size, 
                     padding=depthwise_kernel_size // 2, groups=dim, bias=False),
            nn.BatchNorm2d(dim),
            nn.ReLU(),
        )
        
        self.pointwise = nn.Sequential(
            nn.Conv2d(dim, dim, kernel_size=1, bias=False),
            nn.BatchNorm2d(dim),
            nn.ReLU(),
        )


    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = x
        x = self.depthwise(x)
        x = x + residual
        x = self.pointwise(x)
        return x

B = 4 
C = 256 
H = W = 14 

x = torch.randn(B, C, H, W)
print(f"Input shape: {x.shape}")

block = ConvMixer(
    dim=C,
    depthwise_kernel_size=3,
)

output = block(x)
print(f"\nOutput shape: {output.shape}")

Input shape: torch.Size([4, 256, 14, 14])

Output shape: torch.Size([4, 256, 14, 14])


## Вопрос (0.2 балла)

Объясните, почему MLPMixer, ConvMixer может работать почти так же эффективно, как обычный Multihead Attention.

Напишите формулу, связывающую Multihead Attention, ConvMixer и MLPMixer

Опишите преимущества и недостатки между ConvMixer, MLPMixer и Multihead Attention

---

Ответ: 
MLPMixer, ConvMixer может работать почти так же эффективно, как обычный Multihead Attention, т.к. они все реализуют token-mixing и channel-mixing.<br>
Общая формула следующая:<br> 
   X' = X + token_mixing(Normalize(X))<br>
   X'' = X' + channel_mixing(Normalize(X')), <br>
где:
- Для Multihead Attention: token-mixing - attention, channel mixing - MLP 
- Для ConvMixer: token-mixing - Depthwise convolution, channel mixing - Pointwise convolution
- Для MLPMixer: token-mixing - MLP1, channel mixing - MLP2

Преимущества ConvMixer, MLPMixer:
- Отсутствие квадратичной сложности
- Используют обученные веса
- Меньше параметров, следовательно быстрее в инференсе

Недостатки ConvMixer, MLPMixer:
- Ограничены локальным полем восприятия
- MLPMixer не использует пространственную структуру

Преимущества Multihead Attention:
- Работа с длинными зависимостями
- Более интерпретируемо

Недостатки Multihead Attention:
- Квадратичная сложность 
- Большее число параметров